<!-- SPDX-License-Identifier: GPL-2.0-only -->
# 20 Views

The notebook judges the four views: the directions of the axes of the GZ lesson in each normal view as
`View()` of `3D_bau.frm` sets them up, the reference lines between the associated views, the way back
from a pixel to a point, the edges and outlines a view draws, the grid, the picking of a body, the
oblique view of version 1.3 and the turn of the tab Kamera.

In [ ]:
width = 400
height = 300

In [ ]:
import sys, json, math
sys.path.insert(0, '..')
from harness import *

In [ ]:
build_devtool()

## 1. The axes of the normal views

The front view (Aufriss) shows y to the right and z up, the top view (Grundriss) y to the right and x
down, the side view (Kreuzriss) x to the left and z up, as the lesson draws them.

In [ ]:
def pixel(view, p):
    return devtool('project', view, *p, width, height)

def direction(view, axis):
    o = pixel(view, (0, 0, 0))
    e = pixel(view, axis)
    return (round(e[0] - o[0], 3), round(e[1] - o[1], 3))

ppu = 24.0  # the default scale of Ortho
want = {
    (FRONT, (0, 1, 0)): (ppu, 0), (FRONT, (0, 0, 1)): (0, -ppu), (FRONT, (1, 0, 0)): (0, 0),
    (TOP, (0, 1, 0)): (ppu, 0), (TOP, (1, 0, 0)): (0, ppu), (TOP, (0, 0, 1)): (0, 0),
    (SIDE, (1, 0, 0)): (-ppu, 0), (SIDE, (0, 0, 1)): (0, -ppu), (SIDE, (0, 1, 0)): (0, 0),
}
names = {FRONT: 'Aufriss', TOP: 'Grundriss', SIDE: 'Kreuzriss'}
for (view, axis), expected in want.items():
    got = direction(view, axis)
    check(close(got, expected, 1e-3), f'{names[view]}: the axis {axis} runs {expected} pixels per unit', got)

## 2. The reference lines

The three normal views share one centre and one scale, so a point lies at the same column in the
front and the top view and at the same row in the front and the side view.

In [ ]:
for p in [(1, 2, 3), (-4, 7.5, 0.5), (9, -3, 6)]:
    f, t, s = pixel(FRONT, p), pixel(TOP, p), pixel(SIDE, p)
    check(close(f[0], t[0], 1e-3), f'{p} lies on one vertical reference line in Aufriss and Grundriss', f, t)
    check(close(f[1], s[1], 1e-3), f'{p} lies on one horizontal reference line in Aufriss and Kreuzriss', f, s)

## 3. From a pixel back to a point

A parallel view answers the two coordinates it shows and nothing for the third.

In [ ]:
for view, hidden in [(FRONT, 0), (SIDE, 1), (TOP, 2)]:
    p = (1.5, -2.0, 3.25)
    x, y = pixel(view, p)
    back = devtool('unproject', view, x, y, width, height)
    shown = [i for i in range(3) if i != hidden]
    check(back[hidden] is None and close([back[i] for i in shown], [p[i] for i in shown], 1e-3),
          f'{names[view]} turns the pixel of {p} back into its two shown coordinates', back)

## 4. The edges and the outline

A view draws the edges where faces meet at an angle and the outline where a curved surface turns away
from the viewer. A cube shows its twelve edges, a cylinder seen from the front its two rims and its
two outer generators, and a sphere seen from the front the great circle x = 0.

In [ ]:
cube = devtool('lines', packed(item('CUBE')), FRONT, width, height)
check(cube['styles']['edge'] == 12, 'the front view of a cube draws its twelve edges', cube['styles'])

cylinder = devtool('lines', packed(item('CYLINDER')), FRONT, width, height)
check(cylinder['styles']['edge'] == 2 * 48 + 2, 'the front view of a cylinder draws two rims and two generators', cylinder['styles'])

sphere = devtool('lines', packed(item('SPHERE')), FRONT, width, height)
points = [(s[0], s[1], s[2]) for s in sphere['segments']] + [(s[3], s[4], s[5]) for s in sphere['segments']]
on_circle = all(abs(x) < 1e-3 and abs(math.hypot(y, z) - 1) < 1e-3 for x, y, z in points)
check(sphere['count'] > 40 and on_circle, 'the outline of a sphere in the front view is its great circle x = 0', sphere['count'])

top = devtool('lines', packed(item('SPHERE')), TOP, width, height)
points = [(s[0], s[1], s[2]) for s in top['segments']]
check(all(abs(z) < 1e-3 for _, _, z in points), 'the outline of a sphere in the top view is its equator')

## 5. A body cut away by a difference

The rasterizer does not compute the solid of the CSG, so a body a difference takes away draws as a
dashed ghost in the style `cutter`.

In [ ]:
scene = packed(item('DIFFERENCE'), item('CUBOID', size=(4, 4, 2), parent=0), item('CYLINDER', (2, 2, -1), (1, 1, 4), parent=0))
styles = devtool('lines', scene, FRONT, width, height)['styles']
check(styles['edge'] == 12 and styles['cutter'] == 98, 'the cuboid draws solid and the drill as a ghost', styles)

## 6. The grid

A normal view shows the grid of its plane of projection over its viewport and the axes it shows. The
oblique view shows the ground from zero to ten and the axes with their arrows, as `D3_Achse` does.

In [ ]:
for view, axes in [(FRONT, ('axis_y', 'axis_z')), (TOP, ('axis_x', 'axis_y')), (SIDE, ('axis_x', 'axis_z'))]:
    g = devtool('grid', view, width, height)['styles']
    shown = [a for a in ('axis_x', 'axis_y', 'axis_z') if g[a] > 0]
    check(tuple(shown) == axes and g['minor'] > 0 and g['major'] > 0, f'{names[view]} shows the grid and the axes {axes}', g)
g = devtool('grid', OBLIQUE, width, height)['styles']
check(g['minor'] + g['major'] == 22 and g['axis_x'] == g['axis_y'] == g['axis_z'] == 3, 'the oblique view shows the ground grid and three arrows', g)

## 7. Picking a body

A click picks the body nearest to the viewer under the pixel, and nothing where no body lies.

In [ ]:
scene = packed(item('SPHERE', (2, 2, 1)), item('CUBE', (5, 1.5, 0.5)))
x, y = pixel(FRONT, (2, 2, 1))
check(devtool('pick', scene, FRONT, x, y, width, height) == 1, 'in the front view the cube in front of the sphere wins')
x, y = pixel(SIDE, (2, 2, 1))
check(devtool('pick', scene, SIDE, x, y, width, height) == 0, 'in the side view the sphere is hit')
check(devtool('pick', scene, FRONT, 2, 2, width, height) == -1, 'an empty corner picks nothing')

## 8. The oblique view of version 1.3

`D3_Refresh` of `Module1.bas` draws the 3D view as a cavalier projection: one unit of y runs 40 points
to the right, one of z 40 points up, and one of x, towards the viewer, 20 points to the left and 5
down, around the fixed centre (164, 230) of a picture box of 328 points. The classic look draws its
Schrägriss so, `BAUSTEINE_CAVALIER` gives the devtool that camera at a 3D zoom, and at the start zoom
of 1 a view of 328.27 points shows the numbers of 1.3 exactly.

In [ ]:
cav = {'BAUSTEINE_CAVALIER': '1'}
side = 328.27
o = devtool('project', OBLIQUE, 0, 0, 0, side, side, env=cav)
check(close(o, (164.135, 229.789), 1e-3), 'the origin sits at the centre of D3_Refresh', o)
for axis, step in [((1, 0, 0), (-20, 5)), ((0, 1, 0), (40, 0)), ((0, 0, 1), (0, -40))]:
    e = devtool('project', OBLIQUE, *axis, side, side, env=cav)
    check(close((e[0] - o[0], e[1] - o[1]), step, 1e-3), f'one unit of {axis} runs {step} points', e)
double = devtool('project', OBLIQUE, 0, 1, 0, side, side, env={'BAUSTEINE_CAVALIER': '2'})
check(close(double[0] - o[0], 80, 1e-3), 'twice the 3D zoom doubles the units', double)

## 9. The dot raster of version 1.3

`grid` of `3D_bau.frm` draws a white dot at every unit of a normal view, and in the Aufriss and the
Kreuzriss one more at every half unit of the ground line. At 24 pixels a unit a view of 400 by 300
around the default centre (5, 5, 4) holds 18 by 14 points of the lattice, and the ground line 17 half
steps.

In [ ]:
dots = {view: devtool('classic-raster', view, width, height)['dots'] for view in (FRONT, SIDE, TOP)}
check(dots[TOP] == 18 * 14, 'the Grundriss holds the lattice alone', dots)
check(dots[FRONT] == dots[SIDE] == 18 * 14 + 17, 'the Aufriss and the Kreuzriss add the half steps of the ground line', dots)

## 10. The turn of the tab Kamera

The tab Kamera of 1.3 turns a copy of the scene in the 3D view and leaves the data alone. The
renderer turns the camera the other way, which gives the same picture, so a point turned by the
camera lands where the turned point lands in the camera without a turn.

In [ ]:
import math
turn = (30, -20, 45)
def rotate(p, t):
    x, y, z = p
    for axis, deg in (('z', t[2]), ('y', t[1]), ('x', t[0])):
        c, s = math.cos(math.radians(deg)), math.sin(math.radians(deg))
        if axis == 'x': y, z = c * y - s * z, s * y + c * z
        if axis == 'y': x, z = c * x + s * z, -s * x + c * z
        if axis == 'z': x, y = c * x - s * y, s * x + c * y
    return (x, y, z)
for env in ({}, cav):
    for p in [(1, 0, 0), (0, 2, 0), (1, 2, 3)]:
        turned = devtool('project', OBLIQUE, *p, width, height, env={**env, 'BAUSTEINE_TURN': ','.join(map(str, turn))})
        plain = devtool('project', OBLIQUE, *rotate(p, turn), width, height, env=env)
        check(close(turned, plain, 1e-2), f'the turned camera shows {p} as the turned point {"(cavalier)" if env else "(orbit)"}', turned, plain)